# MAANG Stocks: Returns and Normality Analysis

This notebook downloads one year of daily adjusted close prices for MAANG stocks (Meta, Apple, Amazon, Netflix, Google), computes daily returns and log returns, summarizes mean, standard deviation, skewness, and excess kurtosis, and performs the Jarque-Bera normality test.

In [ ]:
import datetime
import pandas as pd
import numpy as np
import yfinance as yf
from scipy.stats import skew, kurtosis, jarque_bera

# Define the MAANG tickers for analysis
tickers = ['META', 'AAPL', 'AMZN', 'NFLX', 'GOOGL']

# Download one year of daily adjusted close prices
end_date = datetime.date.today()
start_date = end_date - datetime.timedelta(days=365)
prices = yf.download(tickers, start=start_date, end=end_date, progress=False, actions=False)["Close"]
prices = prices.dropna(how='all')

## Statistical Formulas and Definitions

- `data.std(ddof=1)`: unbiased sample standard deviation with Bessel's correction (degrees of freedom = $n-1$):
    $$
    s = \sqrt{\frac{1}{n-1}\sum_{i=1}^n (x_i - \bar{x})^2}\,.
    $$
    See https://pandas.pydata.org/docs/reference/api/pandas.Series.std.html
    
- `skew(data, bias=False)`: unbiased sample skewness. The adjusted estimator is:
    $$
    G_1 = \frac{\sqrt{n(n-1)}}{n-2} \frac{m_3}{m_2^{3/2}}\,,
    $$
    where $m_k = \frac{1}{n}\sum_{i=1}^n (x_i - \bar{x})^k$ is the k-th central moment and $n$ is the sample size. See https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.skew.html

- `kurtosis(data, fisher=True, bias=False)`: unbiased excess kurtosis. The adjusted estimator is:
    $$
    G_2 = \frac{(n-1)}{(n-2)(n-3)}\left((n+1)g_2 + 6\right)\,,
    $$
    where $g_2 = \frac{m_4}{m_2^2} - 3$ is the biased excess kurtosis estimator, and $m_k$ is the k-th central moment. See https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.kurtosis.html

- `jarque_bera(data)`: Jarque-Bera test statistic for normality: 
    $$
    \mathrm{JB} = \frac{n}{6} \left(S^2 + \frac{(K-3)^2}{4} \right)\,,
    $$
    where \(S\) is skewness and \(K\) is kurtosis. Returns `(JB, p-value)`. See https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.jarque_bera.html

In [ ]:
# Compute daily simple returns and log returns
returns = prices.pct_change().dropna()
log_returns = np.log(prices / prices.shift(1)).dropna()

summary = []
for label, series in [('Returns', returns), ('Log Returns', log_returns)]:
    for ticker in tickers:
        data = series[ticker]
        stat = {
            'Ticker': ticker,
            'Type': label,
            'Mean': data.mean(),
            'Std Dev': data.std(ddof=1),
            'Skewness': skew(data, bias=False),
            'Kurtosis': kurtosis(data, fisher=True, bias=False),
        }
        jb_stat, jb_pvalue = jarque_bera(data)
        stat['Jarque-Bera'] = jb_stat
        stat['JB p-value'] = jb_pvalue
        summary.append(stat)

summary_df = pd.DataFrame(summary)
summary_df[['Mean', 'Std Dev', 'Skewness', 'Kurtosis', 'Jarque-Bera', 'JB p-value']] = summary_df[['Mean', 'Std Dev', 'Skewness', 'Kurtosis', 'Jarque-Bera', 'JB p-value']].round(6)
summary_df

## Interpretation

The table above shows the key descriptive statistics and Jarque-Bera test results for both daily simple returns and log returns. 

**Key Findings:**

- **Non-Zero Skewness**: Most MAANG stocks exhibit significant negative or positive skewness, indicating asymmetric return distributions. A normal distribution has zero skewness.
- **Excess Kurtosis**: Positive excess kurtosis indicates **fat tails** — more extreme returns than a normal distribution would predict. This is a common empirical feature of financial asset returns.
- **Jarque-Bera Test**: Low p-values (typically $p < 0.05$) provide strong evidence against normality. Most stock returns reject the null hypothesis of normality.

**Conclusion**: This analysis demonstrates that **asset returns are typically not-normal**, exhibiting skewness and excess kurtosis (heavy tails). This has important implications for risk management and option pricing, as models assuming normality (e.g., Black-Scholes) may underestimate tail risk. The non-normality of returns is one of the most robust empirical findings in financial econometrics.